# ComfyUI no Colab + Google Drive (versão limpa)

**Regra de ouro:** o *código* do ComfyUI fica no disco local do Colab (`/content`, rápido).
Só o que é **pesado e precisa persistir** (modelos, outputs, workflows, inputs) fica no Drive.

Rodar o ComfyUI inteiro de dentro do `/content/drive/...` é o motivo nº1 de lentidão:
o Drive é um filesystem de rede (FUSE), e o Python lê milhares de arquivos pequenos no boot.

Ordem de uso: **Célula 1 → 2 → (3 opcional) → 4**.

In [ ]:
#@title 1. Montar Drive + instalar ComfyUI (local) { display-mode: "form" }
import os, subprocess, pathlib
from google.colab import drive

DRIVE_ROOT = '/content/drive'
DRIVE_DATA = f'{DRIVE_ROOT}/MyDrive/ComfyUI_Data'   #@param {type:"string"}
COMFY      = '/content/ComfyUI'                     # código: disco local

if not os.path.ismount(DRIVE_ROOT):
    drive.mount(DRIVE_ROOT)

def sh(cmd, cwd=None):
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

# --- 1. ComfyUI no disco local (rápido). Se cair a sessão, roda de novo: leva ~1 min.
if not os.path.exists(COMFY):
    sh('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git ' + COMFY)
else:
    sh('git pull', cwd=COMFY)

# --- 2. Estrutura persistente no Drive
MODEL_DIRS = ['checkpoints','loras','vae','clip','clip_vision','controlnet',
              'upscale_models','embeddings','unet','ipadapter','ultralytics/bbox','sams']
for d in MODEL_DIRS:
    pathlib.Path(f'{DRIVE_DATA}/models/{d}').mkdir(parents=True, exist_ok=True)
for d in ['output','input','user']:      # workflows salvos ficam em user/
    pathlib.Path(f'{DRIVE_DATA}/{d}').mkdir(parents=True, exist_ok=True)

# --- 3. Aponta o ComfyUI para os modelos do Drive (sem copiar nada)
yaml = 'drive:\n  base_path: ' + DRIVE_DATA + '/models/\n'
yaml += ''.join(f'  {d}: {d}\n' for d in MODEL_DIRS if '/' not in d)
yaml += '  ultralytics_bbox: ultralytics/bbox\n'
open(f'{COMFY}/extra_model_paths.yaml','w').write(yaml)

# --- 4. Dependências. Torch já vem no Colab: não reinstalar (economiza minutos).
sh('pip install -q -r requirements.txt', cwd=COMFY)

print('\n✅ Célula 1 OK — código em', COMFY, '| dados em', DRIVE_DATA)

In [ ]:
#@title 2. Custom nodes essenciais { display-mode: "form" }
#@markdown Só o núcleo. Depois de subir, use o **Manager** dentro da UI para adicionar
#@markdown o que faltar — e traga de volta para cá só o que você realmente usa.
import os, subprocess
COMFY = '/content/ComfyUI'; CN = f'{COMFY}/custom_nodes'

manager    = True  #@param {type:"boolean"}
rgthree    = True  #@param {type:"boolean"}
kjnodes    = False #@param {type:"boolean"}
controlnet_aux = False #@param {type:"boolean"}
ipadapter  = False #@param {type:"boolean"}
impact_pack= False #@param {type:"boolean"}
ultimate_upscale = False #@param {type:"boolean"}
rembg      = False #@param {type:"boolean"}

REPOS = {
 'manager':          'https://github.com/Comfy-Org/ComfyUI-Manager',
 'rgthree':          'https://github.com/rgthree/rgthree-comfy',
 'kjnodes':          'https://github.com/kijai/ComfyUI-KJNodes',
 'controlnet_aux':   'https://github.com/Fannovel16/comfyui_controlnet_aux',
 'ipadapter':        'https://github.com/cubiq/ComfyUI_IPAdapter_plus',
 'impact_pack':      'https://github.com/ltdrdata/ComfyUI-Impact-Pack',
 'ultimate_upscale': 'https://github.com/ssitu/ComfyUI_UltimateSDUpscale',
 'rembg':            'https://github.com/john-mnz/ComfyUI-Inspyrenet-Rembg',
}

def sh(c, cwd=None):
    print(f'$ {c}'); subprocess.run(c, shell=True, cwd=cwd, check=False)

for key, on in [('manager',manager),('rgthree',rgthree),('kjnodes',kjnodes),
                ('controlnet_aux',controlnet_aux),('ipadapter',ipadapter),
                ('impact_pack',impact_pack),('ultimate_upscale',ultimate_upscale),
                ('rembg',rembg)]:
    if not on: continue
    url = REPOS[key]; dst = f'{CN}/{url.rsplit("/",1)[-1]}'
    if not os.path.exists(dst):
        sh(f'git clone --depth 1 {url} "{dst}"')
    req = f'{dst}/requirements.txt'
    if os.path.exists(req):
        sh(f'pip install -q -r "{req}"')

print('\n✅ Nodes instalados:', sorted(os.listdir(CN)))

In [ ]:
#@title 3. (Opcional) Baixar um checkpoint direto para o Drive { display-mode: "form" }
#@markdown Baixe **uma vez**; nas próximas sessões ele já está lá.
URL = ''  #@param {type:"string"}
PASTA = 'checkpoints'  #@param ["checkpoints","loras","vae","controlnet","upscale_models"]
HF_TOKEN = ''  #@param {type:"string"}

import subprocess
DRIVE_DATA = '/content/drive/MyDrive/ComfyUI_Data'
if URL:
    hdr = f'--header="Authorization: Bearer {HF_TOKEN}" ' if HF_TOKEN else ''
    subprocess.run(f'wget -c {hdr}--content-disposition "{URL}" '
                   f'-P "{DRIVE_DATA}/models/{PASTA}"', shell=True, check=False)
else:
    print('Cole uma URL em URL e rode de novo.')

In [ ]:
#@title 4. Ligar o ComfyUI { display-mode: "form" }
#@markdown `cloudflared` não precisa de conta nem token. Use `ngrok` só se preferir.
TUNEL = 'cloudflared'  #@param ["cloudflared","ngrok"]

import subprocess, threading, re, time, os
COMFY = '/content/ComfyUI'; DRIVE_DATA = '/content/drive/MyDrive/ComfyUI_Data'
PORT = 8188

if TUNEL == 'cloudflared':
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.run('wget -q -O /usr/local/bin/cloudflared '
          'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
          '&& chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
    def tunnel():
        p = subprocess.Popen(['cloudflared','tunnel','--url',f'http://127.0.0.1:{PORT}'],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            m = re.search(r'https://[-\w.]+\.trycloudflare\.com', line)
            if m: print('\n🚀 LINK DE ACESSO:', m.group(0), '\n')
    threading.Thread(target=tunnel, daemon=True).start()
    time.sleep(3)
else:
    import getpass
    subprocess.run('pip install -q pyngrok', shell=True, check=True)
    from pyngrok import ngrok
    ngrok.kill(); ngrok.set_auth_token(getpass.getpass('Ngrok authtoken: '))
    print('\n🚀 LINK DE ACESSO:', ngrok.connect(PORT, 'http').public_url, '\n')

!cd {COMFY} && python main.py \
  --listen 127.0.0.1 --port {PORT} \
  --output-directory "{DRIVE_DATA}/output" \
  --input-directory "{DRIVE_DATA}/input" \
  --user-directory "{DRIVE_DATA}/user" \
  --preview-method auto \
  --disable-auto-launch

## O que mudou em relação à sua versão

| Antes | Agora | Por quê |
|---|---|---|
| ComfyUI clonado **dentro do Drive** | código em `/content`, dados no Drive | boot de minutos → segundos |
| 20 custom nodes fixos | 2 ligados + 6 opcionais por checkbox | menos import, menos conflito de deps |
| ngrok obrigatório (token) | cloudflared sem conta | zero fricção |
| `git clone` completo | `--depth 1` | download bem menor |
| outputs perdidos ao cair a sessão | `--output-directory` no Drive | persistência |
| Manager clonado 2x (célula 1 e 3) | uma vez só | — |

### Nodes removidos e o motivo
- `SeargeSDXL`, `Derfuu`, `NeoGriever`, `masquerade`, `art-venture`, `PixelArt-Detector`,
  `ComfyUI-Logic`, `SaveImageWithMetaData`, `Image-Saver`: só fazem sentido para workflows
  específicos. Instale sob demanda pelo Manager.
- `Crystools`: monitor de VRAM/CPU — bonito, mas fica fazendo polling o tempo todo.
- `Easy-Use`: pacote enorme, puxa muita dependência e costuma conflitar.
- `Impact-Subpack`: só é necessário junto com o Impact Pack (`UltralyticsDetectorProvider`).

### Dicas
- Se um workflow reclamar de node faltando: **Manager → Install Missing Custom Nodes**.
  Depois marque o checkbox correspondente na Célula 2 (ou adicione em `REPOS`) para ele voltar sozinho.
- Não reinstale `torch`: a versão do Colab já é a correta para a GPU.
- Sem GPU? `Ambiente de execução → Alterar tipo de ambiente → T4 GPU`.